In [2]:
import cv2
import numpy as np 
import matplotlib.pyplot as plt
import math 

# img = cv2.imread('Images/Oggy_Home.webp')
img = cv2.imread('Images/Diagonal.jpeg')
#picture is 2D but cv2 warpAffine needs 2x3 matrix so all matrices are 2x3 except translation which is 3x3

degree = 45
x = 10
y = 10
scale = 0.5
scale_matrix = np.array([[scale, 0, 0], 
                        [0, scale, 0]], dtype=np.float32)

rotation_matrix_cv = cv2.getRotationMatrix2D((img.shape[1]//2, img.shape[0]//2), degree, 1)
rotation_matrix = np.array([[math.cos(math.radians(degree)), -math.sin(math.radians(degree)), 0],
                            [math.sin(math.radians(degree)), math.cos(math.radians(degree)), 0]], dtype=np.float32)

translation_matrix = np.array([[1, 0, x],
                               [0, 1, y]], dtype=np.float32)

shear_matrix = np.array([[1, math.tan(math.radians(degree)), 0],
                         [0, 1, 0]], dtype=np.float32)

In [35]:
#the transformation require the output image size hence img.shape[1] and img.shape[0] 
#1 is for width and 0 is for height, since cv2 uses (width, height) instead of (height, width)
img_scaled = cv2.warpAffine(img, scale_matrix, (img.shape[1], img.shape[0]))
cv2.imshow('Scaled Image', img_scaled)
cv2.imwrite('Outputs/Scaled_Image.jpeg', img_scaled)

img_rotated = cv2.warpAffine(img, rotation_matrix, (img.shape[1], img.shape[0]))
cv2.imshow('Rotated Image', img_rotated)    
cv2.imwrite('Outputs/Rotated_Image.jpeg', img_rotated)
img_built_in_rotated = cv2.warpAffine(img, rotation_matrix_cv, (img.shape[1], img.shape[0]))
cv2.imshow('Built-in Rotated Image', img_built_in_rotated)
cv2.imwrite('Outputs/Built-in_Rotated_Image.jpeg', img_built_in_rotated)

img_sheared = cv2.warpAffine(img, shear_matrix, (img.shape[1], img.shape[0]))
cv2.imshow('Sheared Image', img_sheared)
cv2.imwrite('Outputs/Sheared_Image.jpeg', img_sheared)

img_translated = cv2.warpAffine(img, translation_matrix, (img.shape[1], img.shape[0]))
cv2.imshow('Translated Image', img_translated)
cv2.imwrite('Outputs/Translated_Image.jpeg', img_translated)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [38]:
# Global list to store the coordinates you click
# This is the semi automatic points selection
clicked_points = []
# OpenCV will run this function every time your mouse does something
def get_coordinates(event, x, y, flags, param):
    global clicked_points #technically i have no idea why this is needed
    
    # If the user left-clicks:
    if event == cv2.EVENT_LBUTTONDOWN:
        clicked_points.append([x, y])
        print(f"Point captured: ({x}, {y})")
        
        # Draw a little red circle where you clicked 
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1) #funsie and headache
        cv2.imshow("Click 3 Points", img_copy)

# Load the image and make a copy (so we can draw dots on it safely)
img = cv2.imread('Images/Diagonal.jpeg')
img_copy = img.copy() #this is the headache part, circle directly and i had to delete the damn image

# Create a window AND attach the mouse callback to it
cv2.namedWindow("Click 3 Points") #needed for setMouseCallback to work, otherwise it will throw an error
cv2.setMouseCallback("Click 3 Points", get_coordinates)

# Show the image and wait
while True:
    cv2.imshow("Click 3 Points", img_copy)
    
    # Wait for a key press (or until 3 points are clicked)
    if cv2.waitKey(1) & 0xFF == 27 or len(clicked_points) == 3: # 27 is the ESC key
        break

cv2.destroyAllWindows()

# 5. Convert to the exact NumPy format warpAffine needs!
pts1 = np.float32(clicked_points)

affine_matrix = cv2.getAffineTransform(pts1, np.array([[0, 0], [img.shape[1], 0], [0, img.shape[0]]], dtype=np.float32))
img_affine = cv2.warpAffine(img, affine_matrix, (img.shape[1], img.shape[0]))
cv2.imshow('Affine Transformed Image', img_affine)
cv2.imwrite('Outputs/Affine_Transformed_Image_Semi-auto.jpeg', img_affine)
# for affine the logic is suspected to be linear algebra, 
# it's just shifting the points to be the new axis, 
# thus linear properties stay the same
cv2.waitKey(0)
cv2.destroyAllWindows()


Point captured: (514, 136)
Point captured: (656, 144)
Point captured: (619, 327)


In [39]:
#Manual point selection 
img = cv2.imread('Images/Diagonal.jpeg')
#open paints, painful asf
pts1 = np.float32([[100, 100], [200, 100], [100, 200]]) 
#these are the points seen by Paints, the app. can be anything technically just in the image
affine_matrix = cv2.getAffineTransform(pts1, np.array([[0, 0], [img.shape[1], 0], [0, img.shape[0]]], dtype=np.float32))
img_affine = cv2.warpAffine(img, affine_matrix, (img.shape[1], img.shape[0]))
cv2.imshow('Affine Transformed Image', img_affine)  
cv2.imwrite('Outputs/Affine_Transformed_Image_Manual.jpeg', img_affine)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [40]:
# Global list to store the coordinates you click
# This is the semi automatic points selection
# i copied the semi automatic points selection from affine
clicked_points = []
# OpenCV will run this function every time your mouse does something
def get_coordinates(event, x, y, flags, param):
    global clicked_points #technically i have no idea why this is needed
    
    # If the user left-clicks:
    if event == cv2.EVENT_LBUTTONDOWN:
        clicked_points.append([x, y])
        print(f"Point captured: ({x}, {y})")
        
        # Draw a little red circle where you clicked 
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1) #funsie and headache
        cv2.imshow("New Frame", img_copy)

# Load the image and make a copy (so we can draw dots on it safely)
img = cv2.imread('Images/Diagonal.jpeg')
img_copy = img.copy() #this is the headache part, circle directly and i had to delete the damn image

# Create a window AND attach the mouse callback to it
cv2.namedWindow("New Frame") #needed for setMouseCallback to work, otherwise it will throw an error
cv2.setMouseCallback("New Frame", get_coordinates)

# Show the image and wait
while True:
    cv2.imshow("New Frame", img_copy)
    if cv2.waitKey(1) & 0xFF == 27 or len(clicked_points) == 4: # 27 is the ESC key
        break
cv2.destroyAllWindows()

# Convert to the exact NumPy format warpAffine needs!
pts2 = np.float32(clicked_points)

pts1 = np.float32([
    [0, 0], 
    [img.shape[1], 0], 
    [img.shape[1], img.shape[0]], 
    [0, img.shape[0]]
])
# we are changing the perspective of the image, the innitial points are the image corners
# the destination is now the new perspective
# the points selection is clockwise, the order is top-left, top-right, bottom-right, bottom-left
# the closer 2 points are the more convergence towards those points, the more distorted the image will be, the more far apart the points are the more normal the image will be
# the lines are now unable to maintain linear properties like parrallelism

perspective_matrix = cv2.getPerspectiveTransform(pts1, pts2)
img_perspective = cv2.warpPerspective(img, perspective_matrix, (img.shape[1], img.shape[0]))
cv2.imshow('Perspective Transformed Image', img_perspective)
cv2.imwrite('Outputs/Perspective_Transformed_Image_Semi-auto.jpeg', img_perspective)

cv2.waitKey(0)
cv2.destroyAllWindows()

Point captured: (249, 109)
Point captured: (720, 221)
Point captured: (715, 424)
Point captured: (56, 557)


In [41]:
# Manual point selection with weights, the points are the same as the previous one 
# but we will just change perspective by changing the overal dimension of the image
pts2 = np.float32([
    [img.shape[1] * 0.2, img.shape[0] * 0.1],   # Top-Left moves inward and down
    [img.shape[1] * 0.8, img.shape[0] * 0.1],   # Top-Right moves inward and down
    [img.shape[1], img.shape[0]],               # Bottom-Right stays put
    [0, img.shape[0]]                   # Bottom-Left stays put
])

pts1 = np.float32([
    [0, 0], 
    [img.shape[1], 0], 
    [img.shape[1], img.shape[0]], 
    [0, img.shape[0]]
])

img = cv2.imread('Images/Diagonal.jpeg')

perspective_matrix = cv2.getPerspectiveTransform(pts1, pts2)
img_perspective = cv2.warpPerspective(img, perspective_matrix, (img.shape[1], img.shape[0]))
cv2.imshow('Perspective Transformed Image', img_perspective)
cv2.imwrite('Outputs/Perspective_Transformed_Image_Manual.jpeg', img_perspective)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [27]:
# Global variables to store our clicks
points = []
img_copy = None

# The callback function to draw the lines as you click
def draw_polygon(event, x, y, flags, param):
    global points, img_copy
    
    if event == cv2.EVENT_LBUTTONDOWN:
        points.append((x, y))
        
        # Draw a red dot where you clicked
        cv2.circle(img_copy, (x, y), 3, (0, 0, 255), -1)
        
        # If you have at least 2 points, draw a green line between them
        if len(points) > 1:
            cv2.line(img_copy, points[-2], points[-1], (0, 255, 0), 2)
            
        cv2.imshow("Draw Mask", img_copy)

# Load your source image (the one with the object you want to cut out)
src = cv2.imread('Images/Mr_Incredible_29.webp')  
img_copy = src.copy()

# Set up the window and mouse callback
cv2.namedWindow("Draw Mask")
cv2.setMouseCallback("Draw Mask", draw_polygon)

# Wait for the user to finish drawing
while True:
    cv2.imshow("Draw Mask", img_copy)
    key = cv2.waitKey(1) & 0xFF
    if key == 13:  # 13 is the ENTER key
        break
cv2.destroyAllWindows()

# Create a pure black canvas the exact same size as the source image
mask = np.zeros(src.shape, dtype=np.uint8)

# Convert our list of clicked points into the NumPy array format OpenCV needs
pts = np.array([points], dtype=np.int32)

# Fill the polygon you just drew with pure white (255, 255, 255)
cv2.fillPoly(mask, pts, (255, 255, 255))

# Show the final mask to confirm it worked!
cv2.imshow("Your Custom Mask", mask)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [28]:
bg = cv2.imread('Images/Diagonal.jpeg')
result = cv2.seamlessClone(src, bg, mask, (bg.shape[1]//2, bg.shape[0]//2), cv2.NORMAL_CLONE)
cv2.imshow("Seamless Clone Result", result)
cv2.imwrite('Outputs/Seamless_Clone_Result_manual.jpeg', result)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [29]:
src1 = cv2.imread('Images/screaming cowboy.jpg')
# Click and drag a rectangle around your object.
# Press ENTER to confirm the selection.
rect = cv2.selectROI("Select Object", src1, fromCenter=False, showCrosshair=False) 
# this is the automation part, this take a snapshot and wait for the rectangle saved as a x, y, width, height tuple
cv2.destroyWindow("Select Object") #this just turns it off after selection

# GrabCut needs a blank mask, and two empty arrays for its internal math models
# so technically canvas for opencv to draw on
mask = np.zeros(src1.shape[:2], np.uint8) # the mask itself now is just a black image
bgdModel = np.zeros((1, 65), np.float64) # background canvas
fgdModel = np.zeros((1, 65), np.float64) # foreground canvas
# dont know the difference between background and foreground canvas 
# blank arrays to put the result of the math models 

# 5 is the number of iterations (more iterations = slightly better but slower)
# each time it guess the edges and refine through the iterations
# cv2.GC_INIT_WITH_RECT tells it we are giving it a bounding box - the init rect tbh
# the rect is the foreground init and the rest is background init
# at this point which pixels are classified as fore or background 
cv2.grabCut(img, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)

# GrabCut actually outputs a mask with 4 possible values not just black and white mask
# 0: Definite Background
# 1: Definite Foreground
# 2: Probable Background
# 3: Probable Foreground
# We need to combine 1 and 3 into "White" (255), and 0 and 2 into "Black" (0). so now it's a binary mask and thus
# we can use it following the format of opencv, once again idk why opencv coder loves this
binary_mask = np.where((mask == 2) | (mask == 0), 0, 255).astype('uint8')
# the where function is an if for list in python apparently

extracted_object = cv2.bitwise_and(img, img, mask=binary_mask) #show the cutted oout object
# just bitwise and nothing fancy, this is for the visualization part of the assignment 

cv2.imshow('Extracted Object', extracted_object)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [30]:
bg1 = cv2.imread('Images/City1.jpg')
# uses the knowledge of transformation to scale the background because it's big af
# scale = 0.5
# scale_matrix = np.array([[scale, 0, 0], 
#                         [0, scale, 0]], dtype=np.float32)
# bg1_scaled = cv2.warpAffine(bg1, scale_matrix, (bg1.shape[1], bg1.shape[0]))
# bg_copy = bg1_scaled.copy()
bg_copy = bg1.copy()

point = (0, 0) # default point, will be updated by mouse click
def placement_coordinates(event, x, y, flags, param):
    global point
    if event == cv2.EVENT_LBUTTONDOWN:

        point = (x, y)
        cv2.circle(bg_copy, point, 5, (0, 255, 0), -1)
        cv2.imshow("Select Placement", bg_copy)

cv2.namedWindow("Select Placement")
cv2.setMouseCallback("Select Placement", placement_coordinates)
while True:
    cv2.imshow("Select Placement", bg_copy)
    if cv2.waitKey(1) & 0xFF == 13:  # Exit on ENTER key
        break
cv2.destroyAllWindows()

result = cv2.seamlessClone(src1, bg1, binary_mask, point , cv2.NORMAL_CLONE)
cv2.imshow("Seamless Clone Result", result)
cv2.imwrite('Outputs/Seamless_Clone_Result_auto.jpeg', result)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [9]:
src = cv2.imread('Outputs/Forgive_me_please_2.jpeg')     
bg = cv2.imread('bg1.jpg')     

# uses the knowledge of transformation to scale the background because it's big af
scale = 0.2
scale_matrix = np.array([[scale, 0, 0], 
                        [0, scale, 0]], dtype=np.float32)
bg1_scaled = cv2.warpAffine(bg, scale_matrix, (bg.shape[1], bg.shape[0]))

img_copy = bg1_scaled.copy() #this is the headache part, circle directly and i had to delete the damn image
src_rows, src_cols = src.shape[:2]
bg1_rows, bg1_cols = bg1_scaled.shape[:2]

# 2. Source Frame: The 4 corners of your photo
# Top-Left, Top-Right, Bottom-Right, Bottom-Left
pts1 = np.float32([
    [0, 0], 
    [src_cols, 0], 
    [src_cols, src_rows], 
    [0, src_rows]
])

clicked_points = []
# OpenCV will run this function every time your mouse does something
def get_coordinates(event, x, y, flags, param):
    global clicked_points #technically i have no idea why this is needed
    
    # If the user left-clicks:
    if event == cv2.EVENT_LBUTTONDOWN:
        clicked_points.append([x, y])
        print(f"Point captured: ({x}, {y})")
        
        # Draw a little red circle where you clicked 
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1) #funsie and headache
        cv2.imshow("New Frame", img_copy)

# Load the image and make a copy (so we can draw dots on it safely)

# Create a window AND attach the mouse callback to it
cv2.namedWindow("New Frame") #needed for setMouseCallback to work, otherwise it will throw an error
cv2.setMouseCallback("New Frame", get_coordinates)

# Show the image and wait
while True:
    cv2.imshow("New Frame", img_copy)
    if cv2.waitKey(1) & 0xFF == 27 or len(clicked_points) == 4: # 27 is the ESC key
        break
cv2.destroyAllWindows()
    
pts2 = np.float32(clicked_points)

# Calculate Homography and Warp
# Note: The output size MUST be the size of the background image!
matrix = cv2.getPerspectiveTransform(pts1, pts2)
# 2 different image and projective warp can just bridge it
warped_src = cv2.warpPerspective(src, matrix, (bg1_cols, bg1_rows))
#warped the image to fit the perspective of the background now 
#the output is just a picture with black border (big ahh border tbh) so we solve it by cutting a hole in the background and just add them together 


mask = np.zeros((bg1_rows, bg1_cols), dtype=np.uint8)
# We use fillConvexPoly to draw a solid white shape using our 4 destination points
pts2_int = np.int32(pts2) 
#need to convert to int because fillConvexPoly needs that format, 
# the points are in float because of the perspective transform but 
# we just need the coordinates for the mask so it's fine to convert to int
cv2.fillConvexPoly(mask, pts2_int, 255)

#invert the Mask (White becomes Black, Black becomes White)
mask_inv = cv2.bitwise_not(mask)

# Punch the hole in the background image
# This keeps the background everywhere the mask_inv is white, and makes a black hole where it's black
bg_with_hole = cv2.bitwise_and(bg1_scaled, bg1_scaled, mask=mask_inv)

# Extract the warped image using the original mask
# (This just cleans up any stray pixels outside the polygon) due to the float to int conversion
warped_cutout = cv2.bitwise_and(warped_src, warped_src, mask=mask)

# Since the background has a black hole, and the warped cutout is surrounded by black, 
# adding them together fits them perfectly like puzzle pieces.
final_result = cv2.add(bg_with_hole, warped_cutout)

# cv2.imshow("Final Result", final_result)
# cv2.imwrite('Outputs/Final_Result.jpeg', final_result)  

cv2.imshow("Fun only (forgive me if you see this, teacher)", final_result)
cv2.imwrite('Outputs/Forgive_me_please_2.jpeg', final_result)

cv2.waitKey(0)
cv2.destroyAllWindows()

Point captured: (211, 121)
Point captured: (416, 201)
Point captured: (406, 545)
Point captured: (168, 528)


In [18]:
src = cv2.imread('Outputs/Forgive_me_please_2.jpeg')     
bg = cv2.imread('bg1.jpg')     

# uses the knowledge of transformation to scale the background because it's big af
scale = 0.2
scale_matrix = np.array([[scale, 0, 0], 
                        [0, scale, 0]], dtype=np.float32)
bg1_scaled = cv2.warpAffine(bg, scale_matrix, (bg.shape[1], bg.shape[0]))

img_copy = bg1_scaled.copy() #this is the headache part, circle directly and i had to delete the damn image
src_rows, src_cols = src.shape[:2]
bg1_rows, bg1_cols = bg1_scaled.shape[:2]

# 2. Source Frame: The 4 corners of your photo
# Top-Left, Top-Right, Bottom-Right, Bottom-Left
pts1 = np.float32([
    [0, 0], 
    [src_cols, 0], 
    [src_cols, src_rows], 
    [0, src_rows]
])

clicked_points = []
# OpenCV will run this function every time your mouse does something
def get_coordinates(event, x, y, flags, param):
    global clicked_points #technically i have no idea why this is needed
    
    # If the user left-clicks:
    if event == cv2.EVENT_LBUTTONDOWN:
        clicked_points.append([x, y])
        print(f"Point captured: ({x}, {y})")
        
        # Draw a little red circle where you clicked 
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1) #funsie and headache
        cv2.imshow("New Frame", img_copy)

# Load the image and make a copy (so we can draw dots on it safely)

# Create a window AND attach the mouse callback to it
cv2.namedWindow("New Frame") #needed for setMouseCallback to work, otherwise it will throw an error
cv2.setMouseCallback("New Frame", get_coordinates)

# Show the image and wait
while True:
    cv2.imshow("New Frame", img_copy)
    if cv2.waitKey(1) & 0xFF == 27 or len(clicked_points) == 4: # 27 is the ESC key
        break
cv2.destroyAllWindows()
    
pts2 = np.float32(clicked_points)

# Calculate Homography and Warp
# Note: The output size MUST be the size of the background image!
matrix = cv2.getPerspectiveTransform(pts1, pts2)
# 2 different image and projective warp can just bridge it
warped_src = cv2.warpPerspective(src, matrix, (bg1_cols, bg1_rows))
#warped the image to fit the perspective of the background now 
#the output is just a picture with black border (big ahh border tbh) so we solve it by cutting a hole in the background and just add them together 


mask = np.zeros((bg1_rows, bg1_cols), dtype=np.uint8)
# We use fillConvexPoly to draw a solid white shape using our 4 destination points
pts2_int = np.int32(pts2) 
#need to convert to int because fillConvexPoly needs that format, 
# the points are in float because of the perspective transform but 
# we just need the coordinates for the mask so it's fine to convert to int
cv2.fillConvexPoly(mask, pts2_int, 255)

center_x = int(np.mean(pts2[:, 0]))
center_y = int(np.mean(pts2[:, 1]))

final_result = cv2.seamlessClone(warped_src, bg1_scaled, mask, (center_x, center_y), cv2.NORMAL_CLONE)

# cv2.imshow("Final Result", final_result)
# cv2.imwrite('Outputs/Final_Result.jpeg', final_result)  

cv2.imshow("Fun only (forgive me if you see this, teacher)", final_result)
cv2.imwrite('Outputs/Forgive_me_please_Final.jpeg', final_result)

cv2.waitKey(0)
cv2.destroyAllWindows()

Point captured: (219, 133)
Point captured: (407, 208)
Point captured: (397, 522)
Point captured: (188, 513)
